# Reinforced-concrete damage segmentation with YOLO26-seg

**Team:** Name Surname, Name Surname, Name Surname  <!-- TODO: your names -->
**Course:** AI Methods I – Day 4: Neural networks and computer vision (IAAC MaAI 2026/27)
**Track:** B (detection / segmentation): a YOLO26n-seg model fine-tuned to outline five types of damage on photos of reinforced-concrete (RC) structures.

## 0. Introduction and problem definition

### Use case: the three questions

1. **Who would use this model, and for what?** A facility manager or a structural inspector who has hundreds of photos of a building, a parking garage or a bridge (from a walk-through, a drone flight or a tenant report) and has to decide **which elements an engineer must visit first**. The model outlines the damage on every photo and turns the photo set into a ranked list.
2. **What does a mistake cost?** A **missed** exposed or corroded rebar, or crushed concrete (a false negative), can leave a structural problem unseen until the next inspection, so it is the expensive mistake. A **false alarm** (false positive) costs an engineer a minute to look at a photo and dismiss it. So we prefer **recall over precision**, and we choose the confidence threshold with the **F2 score**, which weighs recall twice as much as precision (section 6).
3. **What would be good enough?** Better than what we could do without fine-tuning: (a) the **pretrained** YOLO26n-seg, which only knows the 80 COCO classes and has no damage classes at all, and (b) a **rule-based crack detector** in the spirit of notebook 02 (dark, thin structures found with morphology), compared on crack pixels. For a screening tool, catching most of the serious damage types (rebar exposure, rebar corrosion, crushing) matters more than drawing perfect outlines.

### The data

| | |
|---|---|
| **Name** | Reinforced concrete structure segmentation dataset 2119 (Wang, 2026) |
| **Source** | Mendeley Data, https://data.mendeley.com/datasets/2vkm6k4cfg/1, DOI 10.17632/2vkm6k4cfg.1 |
| **Licence** | CC BY 4.0 (attribution required; derived models allowed) |
| **Size** | 2,119 photos of damaged RC structures, collected by the authors from public datasets and annotated at pixel level |
| **One example** | One photo + one mask with a class number per pixel: 0 = background, 1 = crack, 2 = spalling, 3 = rebar exposure, 4 = rebar corrosion, 5 = crushing |
| **Instances (authors' count)** | crack 1,883 · spalling 282 · rebar exposure 89 · rebar corrosion 832 · crushing 88 |

YOLO-seg needs **instances** (one polygon per object), but the dataset gives a **semantic** mask (one colour per pixel). We convert it: every **connected region** of one colour becomes one polygon of that class (section 1).

💡 **Where to run:** this notebook is meant for **Google Colab with a T4 GPU** (*Runtime → Change runtime type → T4 GPU*). Set `TRAIN = False` to skip training and use the published weights from Hugging Face; everything else then runs on a CPU.

### 0.1 Settings

Everything you may want to change is in this cell. `TRAIN` and `RUN_EXPERIMENT` decide whether the two training runs happen (about 20 + 40 minutes on a T4) or whether the published results are reused.

In [ ]:
TEAM = "group4"                    # as in the Hugging Face repository name
HF_REPO = f"iaac-maai/day4-{TEAM}-rc-damage-segmenter"
GITHUB_URL = f"https://github.com/iaac-maai/Group4-day4-assignment.git"

TRAIN = True            # False: skip training, download the published weights from HF_REPO
RUN_EXPERIMENT = True   # False: skip the second training run (section 5) and read its results from figures/experiments.csv
EPOCHS = 30             # upper limit; early stopping (patience 15) may stop sooner
SEED = 0

### 0.2 Environment

On Colab, the cell clones the GitHub repository (so the relative paths below work) and installs Ultralytics. If the repository is private or not pushed yet, it creates the same folder structure instead; at the end, section 10 zips the outputs for download.

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    repo = Path("/content/repo")
    if not repo.exists():
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")          # never ask for a password
        if subprocess.run(["git", "clone", "-q", GITHUB_URL, str(repo)], env=env).returncode != 0:
            print("Could not clone the repository: working in an empty project folder instead.")
    for d in ["data", "figures", "models", "notebook"]:
        (repo / d).mkdir(parents=True, exist_ok=True)
    os.chdir(repo / "notebook")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.190", "huggingface_hub==2.2.0"])
print("Working directory:", Path.cwd())

In [ ]:
import json, random, shutil, time, urllib.request, zipfile

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import ultralytics
from matplotlib.patches import Patch
from ultralytics import YOLO

DATA_DIR = Path("..") / "data"
FIG_DIR = Path("..") / "figures"
MODEL_DIR = Path("..") / "models"
RAW_DIR = DATA_DIR / "raw"              # the downloaded dataset, as published
YOLO_DIR = DATA_DIR / "yolo"            # our converted dataset in YOLO format
NEW_DIR = DATA_DIR / "new_images"       # photos NOT in the dataset, for section 7
RUNS_DIR = (Path("..") / "runs").resolve()   # absolute, so Ultralytics does not write to a global folder
for d in [DATA_DIR, FIG_DIR, MODEL_DIR, RAW_DIR, NEW_DIR]:
    d.mkdir(parents=True, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print(f"Python {sys.version.split()[0]} | torch {torch.__version__} | ultralytics {ultralytics.__version__} | "
      f"opencv {cv2.__version__} | device: {torch.cuda.get_device_name(0) if DEVICE == 0 else 'CPU'}")

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 130, "font.size": 10})

def save_fig(fig, name, card_name=None):
    """Save a figure to figures/ and, if card_name is given, a copy to models/ for the model card."""
    fig.savefig(FIG_DIR / name, bbox_inches="tight")
    if card_name:
        fig.savefig(MODEL_DIR / card_name, bbox_inches="tight")

### 0.3 Classes

The class order is the order of the model's outputs. `SEVERITY` is our own ranking of how urgent each type is for an inspector; it is only used to rank photos in the application (section 7).

In [ ]:
CLASSES = ["crack", "spalling", "rebar_exposure", "rebar_corrosion", "crushing"]
DRAW_RGB = {1: (230, 25, 25), 2: (30, 190, 30), 3: (230, 0, 230), 4: (30, 90, 255), 5: (255, 200, 0)}   # same hues as the masks
SEVERITY = {"crack": 1, "spalling": 2, "rebar_corrosion": 3, "rebar_exposure": 3, "crushing": 4}
SERIOUS = ["rebar_exposure", "rebar_corrosion", "crushing"]

LEGEND = [Patch(color=np.array(DRAW_RGB[i + 1]) / 255, label=c) for i, c in enumerate(CLASSES)]

## 1. Data and splits

### 1.1 Download

The notebook downloads the whole Mendeley record as one zip and unpacks it, including any archives inside it. If the automatic download fails (Mendeley sometimes changes its links), download it by hand (*Download All* on the dataset page) and put the zip at `data/rc_damage_2119.zip`. Nothing in `data/` is committed.

In [ ]:
MENDELEY_ZIP = "https://data.mendeley.com/public-api/zip/2vkm6k4cfg/download/1"
ZIP_PATH = DATA_DIR / "rc_damage_2119.zip"

def extract_all(folder):
    """Unpack every archive inside folder, repeatedly (archives can contain archives)."""
    while True:
        archives = [p for p in folder.rglob("*") if p.suffix.lower() in {".zip", ".rar", ".7z"}]
        if not archives:
            return
        for a in archives:
            target = a.with_suffix("")
            target.mkdir(exist_ok=True)
            if a.suffix.lower() == ".zip":
                with zipfile.ZipFile(a) as z:
                    z.extractall(target)
            else:   # .rar / .7z: needs the 7z program (pre-installed on Colab; apt install p7zip-full elsewhere)
                subprocess.run(["7z", "x", "-y", f"-o{target}", str(a)], check=True, stdout=subprocess.DEVNULL)
            a.unlink()

if not any(RAW_DIR.rglob("*.*")):
    if not ZIP_PATH.exists():
        print("Downloading the dataset (this can take a few minutes)...")
        try:
            req = urllib.request.Request(MENDELEY_ZIP, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req) as r, open(ZIP_PATH, "wb") as f:
                shutil.copyfileobj(r, f)
        except Exception as e:
            ZIP_PATH.unlink(missing_ok=True)
            raise SystemExit(f"Automatic download failed ({e}).\nDownload the dataset by hand from "
                             f"https://data.mendeley.com/datasets/2vkm6k4cfg/1 and save it as {ZIP_PATH.resolve()}")
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(RAW_DIR)
    extract_all(RAW_DIR)

files = [p for p in RAW_DIR.rglob("*") if p.is_file()]
print(f"{len(files)} files. Extensions:", pd.Series([p.suffix.lower() for p in files]).value_counts().to_dict())
print("Folders:", sorted({str(p.parent.relative_to(RAW_DIR)) for p in files})[:20])

### 1.2 Pair each photo with its mask

We do not rely on folder names, which we cannot be sure of. A file is matched to the other file with the **same name** (ignoring suffixes such as `_mask` or `_label`), and of the two, the **mask** is the one that consists of a few flat colours (`maskness` close to 1), while a photo has many shades.

In [ ]:
import re

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
SUFFIX_RE = re.compile(r"([_\-\s.]?(mask|masks|label|labels|gt|seg|annotation|annot|color|colour))+$")
PREFIX_RE = re.compile(r"^((mask|label|gt|img|image)[_\-\s.]?)+")

def norm_stem(path):
    return PREFIX_RE.sub("", SUFFIX_RE.sub("", Path(path).stem.lower()))

def maskness(path):
    """Share of the pixels covered by the 8 most frequent colours (quantised in steps of 8):
    close to 1 for a colour-coded mask (a few flat colours), much lower for a photo,
    even a gray concrete photo, which has dozens of shades."""
    im = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if im is None:
        return -1.0
    q = (cv2.resize(im, (128, 128), interpolation=cv2.INTER_NEAREST).astype(np.int64) // 8).reshape(-1, 3)
    _, counts = np.unique(q[:, 0] * 1024 + q[:, 1] * 32 + q[:, 2], return_counts=True)
    return float(np.sort(counts)[::-1][:8].sum() / counts.sum())

def find_pairs(root):
    by_stem = {}
    for p in sorted(root.rglob("*")):
        if p.is_file() and p.suffix.lower() in IMG_EXT:
            by_stem.setdefault(norm_stem(p), []).append(p)
    pairs, problems = [], []
    for stem, ps in by_stem.items():
        if len(ps) < 2:
            problems.append((stem, len(ps)))
            continue
        scores = [maskness(p) for p in ps]
        mask = ps[int(np.argmax(scores))]
        photo = ps[int(np.argmin(scores))]
        if max(scores) < 0.95 or max(scores) - min(scores) < 0.02:   # no clear mask among them
            problems.append((stem, len(ps)))
            continue
        pairs.append((photo, mask))
    return pairs, problems

In [ ]:
pairs, problems = find_pairs(RAW_DIR)
print(f"{len(pairs)} photo/mask pairs found; {len(problems)} names without a clear pair.")
if problems:
    print("Examples without a pair:", problems[:10])
assert len(pairs) > 1500, "Far fewer pairs than the 2,119 expected: look at the folder listing above and adapt find_pairs."

### 1.3 From colour masks to classes, and from classes to polygons

**Masks to classes.** The dataset description speaks of colour labels, but the published masks store a **class number per pixel**: 0 = background, 1 = crack, 2 = spalling, 3 = rebar exposure, 4 = rebar corrosion, 5 = crushing. We found this with a diagnostic cell (our first, colour-based version found no damage at all). The function handles both formats. The next cell **checks** the mapping: the number of masks per class must follow the authors' counts (crack most frequent, then corrosion, spalling, and rebar exposure and crushing rarest).

**Classes to polygons.** Every connected region of one class becomes one YOLO polygon (`class x1 y1 x2 y2 ...`, normalised to 0–1). Tiny regions (annotation noise) are dropped. Two consequences to keep in mind: a crack that the annotator drew in one stroke is one instance, while a crack with a gap is two; and a crack network that encloses intact concrete becomes one filled polygon. Both affect the instance counts and the AP of the crack class.

In [ ]:
# The masks of this dataset store a class NUMBER per pixel (0 = background, 1..5 = the classes in
# the order of the dataset description). Some versions use colours instead: we handle both.
HUE_RANGES = {1: [(0, 10), (170, 179)],   # red     -> crack
              2: [(45, 85)],              # green   -> spalling
              3: [(140, 169)],            # magenta -> rebar exposure
              4: [(100, 130)],            # blue    -> rebar corrosion
              5: [(20, 40)]}              # yellow  -> crushing

def mask_to_classmap(mask_bgr):
    """Mask -> array of class ids: 0 background, 1..5 = CLASSES[0..4]."""
    if mask_bgr.ndim == 2:
        m = mask_bgr
    elif np.array_equal(mask_bgr[..., 0], mask_bgr[..., 1]) and np.array_equal(mask_bgr[..., 1], mask_bgr[..., 2]):
        m = mask_bgr[..., 0]                      # gray image: one value per pixel
    else:
        m = None
    if m is not None:                             # class numbers 0..5; anything else (e.g. 255) -> background
        cm = m.astype(np.uint8).copy()
        cm[cm > len(CLASSES)] = 0
        return cm
    hsv = cv2.cvtColor(mask_bgr, cv2.COLOR_BGR2HSV)   # colour masks: read the hue
    h = hsv[..., 0].astype(np.int16)
    colored = (hsv[..., 1] > 100) & (hsv[..., 2] > 60)
    cm = np.zeros(mask_bgr.shape[:2], np.uint8)
    for cid, ranges in HUE_RANGES.items():
        for lo, hi in ranges:
            cm[colored & (h >= lo) & (h <= hi)] = cid
    return cm

def classmap_to_yolo_lines(cm):
    """One normalised polygon per connected region of each class."""
    H, W = cm.shape
    min_area, min_perimeter = max(20.0, 2e-4 * H * W), 0.03 * (H + W)
    lines = []
    for cid in range(1, len(CLASSES) + 1):
        binary = (cm == cid).astype(np.uint8)
        if not binary.any():
            continue
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for cnt in contours:
            # thin cracks have a small area but a long outline: keep a region if either is large enough
            if cv2.contourArea(cnt) < min_area and cv2.arcLength(cnt, True) < min_perimeter:
                continue
            cnt = cv2.approxPolyDP(cnt, 1.0, True).reshape(-1, 2).astype(np.float64)
            if len(cnt) < 3:
                continue
            cnt[:, 0] /= W; cnt[:, 1] /= H
            lines.append(f"{cid - 1} " + " ".join(f"{v:.5f}" for v in cnt.clip(0, 1).ravel()))
    return lines

In [ ]:
# Check the colour assumption on a few masks: which raw colours occur, and which class do they get?
# Check the class mapping: in how many masks does each class occur?
# (dataset authors' instance counts: crack 1,883 · spalling 282 · rebar exposure 89 · rebar corrosion 832 · crushing 88)
from collections import Counter
seen = Counter()
for _, mask in pairs:
    seen.update(np.unique(mask_to_classmap(cv2.imread(str(mask)))).tolist())
for v in sorted(seen):
    print(f"class {v}: in {seen[v]:4d} masks -> {'background' if v == 0 else CLASSES[v - 1]}")
assert len(seen) > 1, "No damage class found in any mask: the mask format is not what mask_to_classmap expects."

### 1.4 Near-duplicates and the split

The 2,119 photos were **collected from several public datasets**, so the same scene can appear more than once (the same photo at another size, a mirrored copy, a slightly different crop). If one copy lands in training and the other in test, the test score is inflated. The dataset does not tell us which source or structure each photo comes from, so we do the next best thing: we compute a **perceptual hash** (dHash) of every photo, also of its mirrored version, and put photos whose hashes differ by at most 6 of 64 bits in the **same group**. Groups are never split.

Then we split the groups **70 / 15 / 15** into training, validation and test, and **stratify** by class: groups that contain the rarest classes (rebar exposure, crushing) are placed first, each into the split that is furthest below its share of that class. This keeps a few examples of every class in validation and test. **The split happens before we look at any model output**, and the test set is used once, in section 6.

To keep training fast, photos are resized so that their longer side is at most 1,280 pixels (masks with nearest-neighbour interpolation, so no new colours appear).

In [ ]:
MAX_SIDE = 1280

def dhash_bits(img_bgr):
    g = cv2.resize(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY), (9, 8), interpolation=cv2.INTER_AREA)
    return (g[:, 1:] > g[:, :-1]).ravel()

def group_near_duplicates(bits, bits_flipped, max_dist=6):
    """Union-find over photos whose hashes (or mirrored hashes) differ by <= max_dist bits."""
    n = len(bits)
    parent = np.arange(n)
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    for i in range(n - 1):
        d = np.minimum((bits[i + 1:] != bits[i]).sum(1), (bits[i + 1:] != bits_flipped[i]).sum(1))
        for j in np.flatnonzero(d <= max_dist) + i + 1:
            ri, rj = find(i), find(j)
            if ri != rj:
                parent[rj] = ri
    return np.array([find(i) for i in range(n)])

def stratified_group_split(presence, groups, fractions=(0.70, 0.15, 0.15), seed=SEED):
    """presence: (n_images x n_classes) 0/1 array. All images of a group get the same split;
    groups with the rarest classes are placed first, into the split that needs that class most."""
    names, f = np.array(["train", "val", "test"]), np.array(fractions)
    gids = np.unique(groups)
    np.random.default_rng(seed).shuffle(gids)
    g_p = np.stack([presence[groups == g].sum(0) for g in gids]).astype(float)
    g_n = np.array([(groups == g).sum() for g in gids], float)
    class_tot = presence.sum(0).clip(min=1).astype(float)
    rarest = np.where(g_p > 0, class_tot, np.inf).min(1)
    tgt_c, tgt_n = f[:, None] * class_tot[None, :], f * len(presence)
    have_c, have_n = np.zeros_like(tgt_c), np.zeros(3)
    split_of = {}
    for k in np.argsort(rarest, kind="stable"):
        present = g_p[k] > 0
        need = ((tgt_c - have_c) / tgt_c)[:, present].sum(1) + (tgt_n - have_n) / tgt_n
        s = int(np.argmax(need))
        split_of[gids[k]] = s
        have_c[s] += g_p[k]; have_n[s] += g_n[k]
    return names[[split_of[g] for g in groups]]

In [ ]:
RECORDS_CSV = YOLO_DIR / "records.csv"
DATA_YAML = YOLO_DIR / "data.yaml"

if RECORDS_CSV.exists() and DATA_YAML.exists():
    records = pd.read_csv(RECORDS_CSV)
    print("Converted dataset found: skipping the conversion.")
else:
    stage = YOLO_DIR / "_stage"
    for d in [stage, YOLO_DIR / "classmaps"]:
        d.mkdir(parents=True, exist_ok=True)
    rows, bits, bits_f = [], [], []
    t0 = time.time()
    for k, (photo, mask) in enumerate(pairs):
        img, m = cv2.imread(str(photo)), cv2.imread(str(mask))
        if img is None or m is None:
            continue
        H, W = img.shape[:2]
        if m.shape[:2] != (H, W):
            m = cv2.resize(m, (W, H), interpolation=cv2.INTER_NEAREST)
        s = min(1.0, MAX_SIDE / max(H, W))
        if s < 1:
            img = cv2.resize(img, (round(W * s), round(H * s)), interpolation=cv2.INTER_AREA)
            m = cv2.resize(m, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
        cm = mask_to_classmap(m)
        lines = classmap_to_yolo_lines(cm)
        iid = f"rc{k:04d}"
        cv2.imwrite(str(stage / f"{iid}.jpg"), img, [cv2.IMWRITE_JPEG_QUALITY, 95])
        (stage / f"{iid}.txt").write_text("\n".join(lines))
        cv2.imwrite(str(YOLO_DIR / "classmaps" / f"{iid}.png"), cm)      # ground truth for pixel-level scores
        inst = np.bincount([int(l.split()[0]) for l in lines], minlength=len(CLASSES))
        pix = np.bincount(cm.ravel(), minlength=len(CLASSES) + 1)[1:] / cm.size
        rows.append({"id": iid, "source": str(photo.relative_to(RAW_DIR)), "width": img.shape[1], "height": img.shape[0],
                     **{f"n_{c}": int(inst[i]) for i, c in enumerate(CLASSES)},
                     **{f"px_{c}": float(pix[i]) for i, c in enumerate(CLASSES)}})
        bits.append(dhash_bits(img)); bits_f.append(dhash_bits(cv2.flip(img, 1)))
        if k % 250 == 0:
            print(f"{k}/{len(pairs)} converted ({time.time() - t0:.0f} s)")
    records = pd.DataFrame(rows)
    records["group"] = group_near_duplicates(np.array(bits), np.array(bits_f))
    presence = (records[[f"n_{c}" for c in CLASSES]].to_numpy() > 0).astype(int)
    records["split"] = stratified_group_split(presence, records["group"].to_numpy())
    for split in ["train", "val", "test"]:
        (YOLO_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
        (YOLO_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)
    for r in records.itertuples():
        shutil.move(stage / f"{r.id}.jpg", YOLO_DIR / "images" / r.split / f"{r.id}.jpg")
        shutil.move(stage / f"{r.id}.txt", YOLO_DIR / "labels" / r.split / f"{r.id}.txt")
    shutil.rmtree(stage)
    records.to_csv(RECORDS_CSV, index=False)
    DATA_YAML.write_text(f"path: {YOLO_DIR.resolve()}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n"
                         + "".join(f"  {i}: {c}\n" for i, c in enumerate(CLASSES)))

n_cols = [f"n_{c}" for c in CLASSES]
dup_groups = records.groupby("group").size()
print(f"{len(records)} images, {(dup_groups > 1).sum()} groups of near-duplicates holding {dup_groups[dup_groups > 1].sum()} images; "
      f"{(records[n_cols].sum(axis=1) == 0).sum()} images without any damage region left after conversion.")
summary = records.groupby("split")[n_cols].sum().rename(columns=lambda c: c[2:])
summary.insert(0, "images", records.groupby("split").size())
summary = summary.loc[["train", "val", "test"]]
summary.loc["total"] = summary.sum()
print("\nImages and instances (polygons) per class and split:")
summary

💡 **Compare with the authors' counts** (crack 1,883 · spalling 282 · rebar exposure 89 · rebar corrosion 832 · crushing 88). Our counts are **regions**, not the authors' instances, so they will be higher where a damage area falls apart into several pieces (typically cracks). If a class has **no** instances at all, the colour mapping of section 1.3 is wrong for that class: check the printed colours.

### 1.5 Look at the data

One photo per class with its mask drawn on top, and the number of instances per class and split. The imbalance is the most important fact about this dataset: rebar exposure and crushing have far fewer examples than cracks.

In [ ]:
def load_image(iid, split=None):
    split = split or records.set_index("id").loc[iid, "split"]
    return cv2.imread(str(YOLO_DIR / "images" / split / f"{iid}.jpg"))

def load_gt(iid):
    return cv2.imread(str(YOLO_DIR / "classmaps" / f"{iid}.png"), cv2.IMREAD_GRAYSCALE)

def overlay(img_bgr, cm, alpha=0.55):
    """Photo (BGR) with the class map drawn on top, returned as RGB."""
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32)
    color = np.zeros_like(rgb)
    for cid, col in DRAW_RGB.items():
        color[cm == cid] = col
    on = cm > 0
    rgb[on] = (1 - alpha) * rgb[on] + alpha * color[on]
    return rgb.astype(np.uint8)

train_rec = records[records.split == "train"]
fig, axes = plt.subplot_mosaic([list("abcde"), ["bar"] * 5], figsize=(16, 8.5), height_ratios=[1, 0.9])
for key, c in zip("abcde", CLASSES):
    # an example where this class covers a large share of the photo
    iid = train_rec.sort_values(f"px_{c}", ascending=False).iloc[min(3, len(train_rec) - 1)]["id"]
    axes[key].imshow(overlay(load_image(iid), load_gt(iid)))
    axes[key].set_title(f"{c}\n({iid})", fontsize=10)
    axes[key].axis("off")
counts = summary.drop(index="total").drop(columns="images").T
counts.plot.bar(ax=axes["bar"], color=["#4c72b0", "#dd8452", "#55a868"], rot=0)
axes["bar"].set_title("Instances (connected regions) per class and split")
axes["bar"].set_xlabel("Class"); axes["bar"].set_ylabel("Number of instances")
for cont in axes["bar"].containers:
    axes["bar"].bar_label(cont, fontsize=8)
fig.legend(handles=LEGEND, loc="upper center", ncol=5, bbox_to_anchor=(0.5, 1.02))
fig.suptitle("Dataset: examples with their masks, and class balance", y=1.06, fontsize=13)
save_fig(fig, "01_dataset_overview.png", "dataset_overview.png")
plt.show()

## 2. Baselines

### 2.1 The pretrained model without fine-tuning

`yolo26n-seg.pt` was trained on COCO: people, cars, animals, furniture... It has **no class for any type of damage**, so its mAP on our classes is **0 by construction**. The question is only whether it finds anything at all on our photos, and whether that is useful.

In [ ]:
coco = YOLO("yolo26n-seg.pt")
val_ids = records.loc[records.split == "val", "id"].tolist()
sample = random.Random(SEED).sample(val_ids, 12)
found = []
for iid, r in zip(sample, coco.predict([str(YOLO_DIR / "images" / "val" / f"{i}.jpg") for i in sample],
                                       conf=0.25, verbose=False, device=DEVICE, stream=True)):
    found.append({"image": iid, "objects found": ", ".join(coco.names[int(c)] for c in r.boxes.cls) or "-"})
pd.DataFrame(found)

*Reading:* write here what the pretrained model finds (typically nothing, or an occasional "person", "car" or "train" in the background). It cannot tell a crack from intact concrete: fine-tuning is needed.

### 2.2 A rule-based crack detector (notebook 02)

Cracks are **dark, thin lines**. The black-hat transform (closing minus the image) highlights dark structures that are thinner than its kernel; Otsu's threshold turns that into a binary mask, and we keep only elongated regions. This is a fair baseline for the **crack** class only: the other damage types have no simple shape or colour rule. We score it **per pixel** (precision, recall and IoU of crack pixels), because it produces pixels, not instances.

In [ ]:
def rule_based_crack(img_bgr):
    g = cv2.GaussianBlur(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY), (3, 3), 0)
    k = max(7, round(min(g.shape) / 60)) | 1
    bh = cv2.morphologyEx(g, cv2.MORPH_BLACKHAT, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k)))
    _, m = cv2.threshold(bh, 0, 1, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    n, lab, stats, _ = cv2.connectedComponentsWithStats(m.astype(np.uint8), connectivity=8)
    longest = stats[:, 2:4].max(1)
    keep = (stats[:, 4] >= 30) & (longest >= 0.05 * max(g.shape))     # large and elongated enough
    keep[0] = False
    return keep[lab]

def pixel_counts(pred, gt):
    return np.array([(pred & gt).sum(), (pred & ~gt).sum(), (~pred & gt).sum()], dtype=np.int64)

def pixel_scores(tp, fp, fn):
    return {"precision": tp / max(tp + fp, 1), "recall": tp / max(tp + fn, 1), "IoU": tp / max(tp + fp + fn, 1)}

def rule_based_eval(ids):
    tot = np.zeros(3, np.int64)
    for iid in ids:
        tot += pixel_counts(rule_based_crack(load_image(iid)), load_gt(iid) == 1)
    return pixel_scores(*tot)

crack_val_ids = records.loc[(records.split == "val") & (records.n_crack > 0), "id"].tolist()
rb_val = rule_based_eval(crack_val_ids)
print(f"Rule-based crack detector on {len(crack_val_ids)} validation photos with cracks: "
      + ", ".join(f"{k} {v:.3f}" for k, v in rb_val.items()))

In [ ]:
show = crack_val_ids[:: max(1, len(crack_val_ids) // 3)][:3]
fig, axes = plt.subplots(len(show), 3, figsize=(12, 3.6 * len(show)))
for row, iid in zip(np.atleast_2d(axes), show):
    img, gt = load_image(iid), load_gt(iid)
    row[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); row[0].set_title(f"{iid}: photo")
    row[1].imshow(overlay(img, (gt == 1).astype(np.uint8))); row[1].set_title("True crack pixels")
    row[2].imshow(overlay(img, rule_based_crack(img).astype(np.uint8))); row[2].set_title("Rule-based detector")
    for a in row:
        a.axis("off")
fig.suptitle("Baseline: rule-based crack detection (black-hat + Otsu) on validation photos", fontsize=13)
fig.tight_layout()
save_fig(fig, "02_rule_based_baseline.png")
plt.show()

*Reading:* write here what the rule-based detector gets right (clean, high-contrast cracks) and wrong (it also marks joints, shadows, stains and rebar as "cracks", and misses faint cracks). This is the number to beat for the crack class.

## 3. The model and its training

We fine-tune **YOLO26n-seg** (the nano segmentation model, about 3 M parameters, pretrained on COCO): it predicts, for every object, a box, a class, a confidence and a mask. Settings:

- **Image size 640**, batch 16, up to `EPOCHS` epochs with **early stopping** (patience 15 epochs without improvement on validation).
- Ultralytics' default augmentation (mosaic, flips, scale, HSV colour changes) and optimiser (chosen automatically: AdamW for short runs).
- Fixed seed and `deterministic=True`.
- **Tier:** free cloud (Colab T4 GPU). On a CPU, use `TRAIN = False`.

Ultralytics keeps the weights of the epoch with the best validation fitness (`best.pt`): that is our model selection. The test set is not involved.

In [ ]:
def train_run(name, imgsz, batch, epochs=EPOCHS):
    t0 = time.time()
    model = YOLO("yolo26n-seg.pt")
    model.train(data=str(DATA_YAML.resolve()), epochs=epochs, imgsz=imgsz, batch=batch, patience=15,
                seed=SEED, deterministic=True, project=str(RUNS_DIR), name=name, exist_ok=True,
                device=DEVICE, workers=2, plots=True, verbose=False)
    run_dir = Path(model.trainer.save_dir)
    return {"name": name, "imgsz": imgsz, "dir": run_dir, "weights": run_dir / "weights" / "best.pt",
            "results": run_dir / "results.csv", "minutes": (time.time() - t0) / 60}

RUN_A = {"name": "seg_n_640", "imgsz": 640, "batch": 16}
if TRAIN:
    run_a = train_run(**RUN_A)
    print(f"Run A finished in {run_a['minutes']:.0f} min: {run_a['weights']}")
else:
    from huggingface_hub import hf_hub_download
    cfg = json.loads(Path(hf_hub_download(HF_REPO, "config.json")).read_text())
    run_a = {"name": "published", "imgsz": cfg["imgsz"], "weights": Path(hf_hub_download(HF_REPO, "model.pt")),
             "results": Path(hf_hub_download(HF_REPO, "training_log.csv")), "minutes": cfg.get("train_minutes", np.nan)}
    print("Using the published model:", HF_REPO, cfg)

In [ ]:
def plot_training_curves(results_csv, title):
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    ep = df["epoch"]
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.3))
    for part, col in [("seg", "C0"), ("box", "C1"), ("cls", "C2")]:
        if f"train/{part}_loss" in df:
            ax[0].plot(ep, df[f"train/{part}_loss"], color=col, label=f"{part} loss, training")
            ax[0].plot(ep, df[f"val/{part}_loss"], color=col, ls="--", label=f"{part} loss, validation")
    ax[0].set_title("Losses"); ax[0].set_ylabel("Loss")
    for key, lab in [("metrics/mAP50(M)", "mask mAP50"), ("metrics/mAP50-95(M)", "mask mAP50-95"), ("metrics/mAP50(B)", "box mAP50")]:
        if key in df:
            ax[1].plot(ep, df[key], label=lab)
    ax[1].set_title("Validation mAP"); ax[1].set_ylabel("mAP")
    for key, lab in [("metrics/precision(M)", "mask precision"), ("metrics/recall(M)", "mask recall")]:
        if key in df:
            ax[2].plot(ep, df[key], label=lab)
    ax[2].set_title("Validation precision and recall (at the best-F1 threshold)"); ax[2].set_ylabel("Score")
    if "metrics/mAP50-95(M)" in df:
        best = df["metrics/mAP50-95(M)"].idxmax()
        for a in ax[1:]:
            a.axvline(ep[best], color="gray", ls=":", label=f"best epoch ({ep[best]})")
    for a in ax:
        a.set_xlabel("Epoch"); a.legend(fontsize=8); a.grid(alpha=0.3)
    fig.suptitle(title, fontsize=13)
    fig.tight_layout()
    return fig

fig = plot_training_curves(run_a["results"], f"Training of run A (YOLO26n-seg, image size {run_a['imgsz']})")
save_fig(fig, "03a_training_curves_run_a.png")
plt.show()

*Reading (fill in after the run):* do the validation losses still go down at the end, or do they flatten or rise while the training losses keep falling (overfitting)? At which epoch is the best validation mAP, and did early stopping end the run? Would more epochs help?

## 4. Look inside and diagnose (validation set)

Before touching the test set, we look at **where the model fails on validation**: the per-class AP, the confusion matrix and a gallery of the photos with the worst predictions.

In [ ]:
def evaluate(weights, split, imgsz, name):
    return YOLO(str(weights)).val(data=str(DATA_YAML.resolve()), split=split, imgsz=imgsz, batch=16, device=DEVICE,
                                  project=str(RUNS_DIR), name=name, exist_ok=True, plots=False, verbose=False)

def per_class_table(m):
    idx = [int(i) for i in m.ap_class_index]
    return pd.DataFrame({"class": [CLASSES[i] for i in idx], "box AP50": m.box.ap50, "mask AP50": m.seg.ap50,
                         "mask AP50-95": m.seg.ap, "mask precision": m.seg.p, "mask recall": m.seg.r}).round(3)

def plot_confusion(m, title):
    cmat = getattr(m, "confusion_matrix", None)
    if cmat is None:
        return None
    mat = np.array(cmat.matrix, dtype=float)          # rows: predicted, columns: true (Ultralytics convention)
    norm = mat / mat.sum(0, keepdims=True).clip(min=1)
    labels = CLASSES + ["background"]
    fig, ax = plt.subplots(figsize=(7.5, 6.5))
    im = ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
    for i in range(len(labels)):
        for j in range(len(labels)):
            if mat[i, j] > 0:
                ax.text(j, i, f"{norm[i, j]:.2f}\n({int(mat[i, j])})", ha="center", va="center", fontsize=8,
                        color="white" if norm[i, j] > 0.5 else "black")
    ax.set_xticks(range(len(labels)), labels, rotation=35, ha="right"); ax.set_yticks(range(len(labels)), labels)
    ax.set_xlabel("True class"); ax.set_ylabel("Predicted class")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label="Share of the true class (column)")
    fig.tight_layout()
    return fig

val_a = evaluate(run_a["weights"], "val", run_a["imgsz"], "val_run_a")
print(f"Run A on validation: mask mAP50 {val_a.seg.map50:.3f}, mask mAP50-95 {val_a.seg.map:.3f}, box mAP50 {val_a.box.map50:.3f}")
per_class_table(val_a)

In [ ]:
fig = plot_confusion(val_a, "Confusion matrix, run A, validation set")
if fig:
    save_fig(fig, "04a_confusion_matrix_val.png"); plt.show()

The gallery below is built from **pixel-level** scores: for each validation photo we draw the predicted masks (confidence ≥ 0.25) into a class map and compare it with the true class map. Per photo, the score is the mean IoU over the classes that are present in the truth or in the prediction. The photos with the lowest score are shown, with the classes the model **missed** (in the truth, not predicted: false negatives) and the classes it **invented** (predicted, not in the truth: false positives).

In [ ]:
def result_to_classmap(r, shape):
    cm = np.zeros(shape, np.uint8)
    if r.masks is None or len(r.boxes) == 0:
        return cm
    order = np.argsort(r.boxes.conf.cpu().numpy())            # most confident drawn last (on top)
    for i in order:
        poly = r.masks.xy[i]
        if len(poly) >= 3:
            cv2.fillPoly(cm, [poly.astype(np.int32)], int(r.boxes.cls[i]) + 1)
    return cm

def pixel_eval(weights, ids, split, imgsz, conf):
    """Per-photo pixel counts (tp, fp, fn) for every class, without keeping the maps in memory."""
    model = YOLO(str(weights))
    paths = [str(YOLO_DIR / "images" / split / f"{i}.jpg") for i in ids]
    rows = []
    for iid, r in zip(ids, model.predict(paths, imgsz=imgsz, conf=conf, verbose=False, device=DEVICE, stream=True)):
        gt = load_gt(iid)
        pred = result_to_classmap(r, gt.shape)
        row = {"id": iid}
        ious = []
        for cid, c in enumerate(CLASSES, start=1):
            tp, fp, fn = pixel_counts(pred == cid, gt == cid)
            row.update({f"tp_{c}": tp, f"fp_{c}": fp, f"fn_{c}": fn})
            if tp + fp + fn > 0:
                ious.append(tp / (tp + fp + fn))
        row["score"] = np.mean(ious) if ious else 1.0
        row["missed"] = ", ".join(c for c in CLASSES if row[f"fn_{c}"] > 0 and row[f"tp_{c}"] == 0)
        row["false alarm"] = ", ".join(c for c in CLASSES if row[f"fp_{c}"] > 0 and row[f"tp_{c}"] == 0)
        rows.append(row)
    return pd.DataFrame(rows)

def pixel_summary(pe):
    out = {}
    for c in CLASSES:
        out[c] = pixel_scores(pe[f"tp_{c}"].sum(), pe[f"fp_{c}"].sum(), pe[f"fn_{c}"].sum())
    return pd.DataFrame(out).T.round(3)

def gallery(weights, ids, split, imgsz, conf, title, pe=None):
    model = YOLO(str(weights))
    fig, axes = plt.subplots(len(ids), 3, figsize=(13, 3.7 * len(ids)))
    for row, iid in zip(np.atleast_2d(axes), ids):
        img, gt = load_image(iid, split), load_gt(iid)
        r = model.predict(str(YOLO_DIR / "images" / split / f"{iid}.jpg"), imgsz=imgsz, conf=conf, verbose=False, device=DEVICE)[0]
        pred = result_to_classmap(r, gt.shape)
        row[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); row[0].set_title(f"{iid}: photo")
        row[1].imshow(overlay(img, gt)); row[1].set_title("Ground truth")
        info = ""
        if pe is not None:
            p = pe.set_index("id").loc[iid]
            info = f"mean IoU {p['score']:.2f}" + (f"\nmissed: {p['missed']}" if p["missed"] else "") \
                   + (f"\nfalse alarm: {p['false alarm']}" if p["false alarm"] else "")
        row[2].imshow(overlay(img, pred)); row[2].set_title(f"Prediction (conf ≥ {conf:.2f})\n{info}", fontsize=9)
        for a in row:
            a.axis("off")
    fig.legend(handles=LEGEND, loc="upper center", ncol=5, bbox_to_anchor=(0.5, 1.0))
    fig.suptitle(title, fontsize=13, y=1.02)
    fig.tight_layout()
    return fig

pe_val_a = pixel_eval(run_a["weights"], val_ids, "val", run_a["imgsz"], conf=0.25)
print("Pixel-level scores of run A on validation (conf 0.25):")
display(pixel_summary(pe_val_a))
worst = pe_val_a.sort_values("score").head(6)["id"].tolist()
fig = gallery(run_a["weights"], worst, "val", run_a["imgsz"], 0.25, "Run A: the six worst validation photos", pe_val_a)
save_fig(fig, "04b_val_failures.png")
plt.show()

*Reading (your own, after the run):* what do the failures have in common? Things to look for:
- **Class confusion** between visually similar types: rebar **exposure** vs. rebar **corrosion** (both show steel; the difference is rust), spalling vs. crushing.
- **Thin cracks**: at 640 pixels a hairline crack is one pixel wide or less, so the mask is too coarse, or the crack is missed.
- **Annotation choices**: regions the annotator left out (or a crack network drawn as one region) count as false alarms (or misses) even when the model is arguably right.
- Lighting, distance, image quality, and the rare classes with few training examples.

## 5. Tuning: does a higher image size help?

The diagnosis suggests that thin cracks and small rebar regions lose detail at 640 pixels. **One change:** we train the same model at **image size 960** (batch 8, so it fits on a T4) and compare both runs on the **validation** set. Everything else stays the same. The cost: roughly twice the training and prediction time.

In [ ]:
RUN_B = {"name": "seg_n_960", "imgsz": 960, "batch": 8}
EXP_CSV = FIG_DIR / "experiments.csv"

def summarise(run, m):
    df = pd.read_csv(run["results"]); df.columns = df.columns.str.strip()
    return {"run": run["name"], "image size": run["imgsz"], "epochs run": len(df),
            "best epoch": int(df.loc[df["metrics/mAP50-95(M)"].idxmax(), "epoch"]),
            "val mask mAP50": round(m.seg.map50, 3), "val mask mAP50-95": round(m.seg.map, 3),
            "val box mAP50": round(m.box.map50, 3), "training time (min)": round(run["minutes"], 1),
            **{f"val mask AP50 {c}": round(float(a), 3) for c, a in zip([CLASSES[int(i)] for i in m.ap_class_index], m.seg.ap50)}}

if TRAIN and RUN_EXPERIMENT:
    run_b = train_run(**RUN_B)
    val_b = evaluate(run_b["weights"], "val", run_b["imgsz"], "val_run_b")
    exp = pd.DataFrame([summarise(run_a, val_a), summarise(run_b, val_b)])
    exp.to_csv(EXP_CSV, index=False)
elif EXP_CSV.exists():
    run_b = None
    exp = pd.read_csv(EXP_CSV)
    print("Experiment results read from", EXP_CSV)
else:
    run_b, exp = None, pd.DataFrame([summarise(run_a, val_a)])
exp.set_index("run").T

In [ ]:
if TRAIN and RUN_EXPERIMENT:
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
    for run, col in [(run_a, "C0"), (run_b, "C3")]:
        df = pd.read_csv(run["results"]); df.columns = df.columns.str.strip()
        ax[0].plot(df["epoch"], df["metrics/mAP50-95(M)"], color=col, label=f"image size {run['imgsz']}")
        ax[0].plot(df["epoch"], df["metrics/mAP50(M)"], color=col, ls="--", alpha=0.6, label=f"image size {run['imgsz']} (mAP50)")
    ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Validation mask mAP"); ax[0].set_title("Validation mask mAP50-95 (solid) and mAP50 (dashed)")
    ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
    ap_cols = [c for c in exp.columns if c.startswith("val mask AP50 ")]
    exp.set_index("image size")[ap_cols].rename(columns=lambda c: c.replace("val mask AP50 ", "")).T.plot.bar(ax=ax[1], rot=20, color=["C0", "C3"])
    ax[1].set_xlabel("Class"); ax[1].set_ylabel("Validation mask AP50"); ax[1].set_title("Per-class validation mask AP50"); ax[1].legend(title="Image size")
    fig.suptitle("Experiment: image size 640 vs. 960 (validation set)", fontsize=13)
    fig.tight_layout()
    save_fig(fig, "05_experiment_image_size.png")
    plt.show()

**Choice of the final model:** the run with the higher **validation mask mAP50-95** (our main metric: it rewards outlines that fit tightly, which matters for measuring damage). If the difference is small (< 0.01), we keep the cheaper 640 model. *Reading (fill in):* did 960 help, and for which classes? Was it worth twice the time?

In [ ]:
if run_b is not None:
    gain = exp.loc[1, "val mask mAP50-95"] - exp.loc[0, "val mask mAP50-95"]
    final = run_b if gain >= 0.01 else run_a
else:
    final = run_a
FINAL_WEIGHTS, FINAL_IMGSZ = final["weights"], final["imgsz"]
print(f"Final model: {final['name']} (image size {FINAL_IMGSZ})")

fig = plot_training_curves(final["results"], f"Training curves of the final model (YOLO26n-seg, image size {FINAL_IMGSZ})")
save_fig(fig, "03_training_curves.png", "training_curves.png")
plt.show()

## 6. Test set, once

### 6.1 Choosing the confidence threshold (on validation)

The threshold is a decision about the **cost of mistakes**, so we choose it on **validation**, not on test. Ultralytics gives the precision and recall of each class for every threshold from 0 to 1. We average them over the classes and pick the threshold with the highest **F2 score**, which weighs recall twice as much as precision: a missed rebar costs more than a photo an engineer has to dismiss (section 0).

In [ ]:
val_f = evaluate(FINAL_WEIGHTS, "val", FINAL_IMGSZ, "val_final")
px = np.asarray(val_f.seg.px)
P, R = np.asarray(val_f.seg.p_curve).mean(0), np.asarray(val_f.seg.r_curve).mean(0)
F1 = 2 * P * R / np.clip(P + R, 1e-9, None)
F2 = 5 * P * R / np.clip(4 * P + R, 1e-9, None)
ok = (px >= 0.05) & (px <= 0.90)
CONF = float(np.round(px[ok][np.argmax(F2[ok])], 2))
i_c = np.abs(px - CONF).argmin()
print(f"Recommended threshold: {CONF:.2f} -> validation precision {P[i_c]:.2f}, recall {R[i_c]:.2f}, F2 {F2[i_c]:.2f}")

fig, ax = plt.subplots(1, 2, figsize=(14, 4.3))
ax[0].plot(px, P, label="precision"); ax[0].plot(px, R, label="recall")
ax[0].plot(px, F1, label="F1", alpha=0.7); ax[0].plot(px, F2, label="F2 (recall counts double)", lw=2.5)
ax[0].axvline(CONF, color="k", ls="--", label=f"recommended: {CONF:.2f}")
ax[0].set_title("Mean over the classes"); ax[0].set_xlabel("Confidence threshold"); ax[0].set_ylabel("Score (mask, validation)")
names_v = [CLASSES[int(i)] for i in val_f.ap_class_index]
for k, c in enumerate(names_v):
    ax[1].plot(px, np.asarray(val_f.seg.r_curve)[k], color=np.array(DRAW_RGB[CLASSES.index(c) + 1]) / 255, label=c)
ax[1].axvline(CONF, color="k", ls="--")
ax[1].set_title("Recall per class"); ax[1].set_xlabel("Confidence threshold"); ax[1].set_ylabel("Recall (mask, validation)")
for a in ax:
    a.legend(fontsize=8); a.grid(alpha=0.3); a.set_xlim(0, 1); a.set_ylim(0, 1.02)
fig.suptitle("Choosing the confidence threshold on the validation set", fontsize=13)
fig.tight_layout()
save_fig(fig, "06_threshold_sweep.png", "threshold_sweep.png")
plt.show()

### 6.2 Test metrics, next to the baselines

Now, and only now, the test set. mAP50 and mAP50-95 do not depend on a threshold; precision and recall are given **at the recommended threshold**. Pixel-level scores compare the model with the rule-based crack detector on exactly the same test photos.

In [ ]:
test_m = evaluate(FINAL_WEIGHTS, "test", FINAL_IMGSZ, "test_final")
tpx = np.asarray(test_m.seg.px); j = np.abs(tpx - CONF).argmin()
test_P, test_R = float(np.asarray(test_m.seg.p_curve).mean(0)[j]), float(np.asarray(test_m.seg.r_curve).mean(0)[j])

test_ids = records.loc[records.split == "test", "id"].tolist()
crack_test_ids = records.loc[(records.split == "test") & (records.n_crack > 0), "id"].tolist()
pe_test = pixel_eval(FINAL_WEIGHTS, test_ids, "test", FINAL_IMGSZ, CONF)
pix_test = pixel_summary(pe_test)
pe_crack = pe_test[pe_test.id.isin(crack_test_ids)]
model_crack = pixel_scores(pe_crack["tp_crack"].sum(), pe_crack["fp_crack"].sum(), pe_crack["fn_crack"].sum())
rb_test = rule_based_eval(crack_test_ids)

overall = pd.DataFrame({
    "Pretrained YOLO26n-seg (COCO, no fine-tuning)": {"mask mAP50": 0.0, "mask mAP50-95": 0.0, "box mAP50": 0.0,
        f"mask precision @ {CONF}": np.nan, f"mask recall @ {CONF}": np.nan, "crack pixel IoU": np.nan},
    "Rule-based crack detector": {"mask mAP50": np.nan, "mask mAP50-95": np.nan, "box mAP50": np.nan,
        f"mask precision @ {CONF}": np.nan, f"mask recall @ {CONF}": np.nan, "crack pixel IoU": rb_test["IoU"]},
    "This model (fine-tuned)": {"mask mAP50": test_m.seg.map50, "mask mAP50-95": test_m.seg.map, "box mAP50": test_m.box.map50,
        f"mask precision @ {CONF}": test_P, f"mask recall @ {CONF}": test_R, "crack pixel IoU": model_crack["IoU"]},
}).round(3)
print(f"Test set: {len(test_ids)} photos. Crack pixel scores on the {len(crack_test_ids)} test photos with cracks: "
      f"rule-based {', '.join(f'{k} {v:.3f}' for k, v in rb_test.items())} | model {', '.join(f'{k} {v:.3f}' for k, v in model_crack.items())}")
overall

In [ ]:
per_class_test = per_class_table(test_m)
per_class_test["test instances"] = [int(summary.loc["test", c]) for c in per_class_test["class"]]
per_class_test = per_class_test.merge(pix_test.add_prefix("pixel ").reset_index(names="class"), on="class")
per_class_test

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
xs = np.arange(len(per_class_test))
for k, (col, lab) in enumerate([("mask AP50", "mask AP50"), ("mask AP50-95", "mask AP50-95"), ("box AP50", "box AP50")]):
    bars = ax.bar(xs + (k - 1) * 0.27, per_class_test[col], 0.27, label=lab)
    ax.bar_label(bars, fmt="%.2f", fontsize=7)
ax.set_xticks(xs, [f"{c}\n(n = {n})" for c, n in zip(per_class_test["class"], per_class_test["test instances"])])
ax.set_ylim(0, 1); ax.set_xlabel("Class (number of test instances)"); ax.set_ylabel("Average precision (test)")
ax.set_title(f"Per-class average precision on the test set (overall mask mAP50 {test_m.seg.map50:.2f})")
ax.legend(); ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
save_fig(fig, "07_per_class_ap.png", "per_class_ap.png")
plt.show()

fig = plot_confusion(test_m, "Confusion matrix, final model, test set")
if fig:
    save_fig(fig, "08_confusion_matrix.png", "confusion_matrix.png"); plt.show()

### 6.3 What it does: predictions next to the truth (test set)

Two of the best, two average and two of the worst test photos, so that the reader sees both what the model does well and how it fails.

In [ ]:
ranked = pe_test.sort_values("score", ascending=False)["id"].tolist()
mid = len(ranked) // 2
show = ranked[:2] + ranked[mid:mid + 2] + ranked[-2:]
fig = gallery(FINAL_WEIGHTS, show, "test", FINAL_IMGSZ, CONF,
              "Test photos: ground truth vs. prediction (rows 1-2: best, 3-4: typical, 5-6: worst)", pe_test)
save_fig(fig, "09_test_predictions.png", "predictions.png")
plt.show()

fig = gallery(FINAL_WEIGHTS, ranked[-6:][::-1], "test", FINAL_IMGSZ, CONF, "The six worst test photos", pe_test)
save_fig(fig, "09b_test_failures.png", "failures.png")
plt.show()

*Reading (fill in):* compare with the baselines (the pretrained model finds no damage at all; is the model's crack IoU clearly above the rule-based one?), name the strongest and the weakest class, and say whether the serious classes reach a useful recall at the recommended threshold.

## 7. Application: ranking new photos for an inspector

The use case: a stack of photos that were **not in the dataset**. Put your own photos (or openly licensed ones, e.g. from Wikimedia Commons, with their licence noted) of concrete structures into `data/new_images/`. The model outlines the damage, and we turn that into an **inspection list**: for every photo, the damage types found, the number of regions and the share of the photo they cover, ranked by **severity first** (crushing > rebar exposure/corrosion > spalling > cracks) and then by damaged area.

⚠️ The damaged area is a share of the **photo**, not of the element: without a scale in the photo (a ruler, a known rebar diameter) it cannot be turned into square metres.

In [ ]:
new_paths = sorted(p for p in NEW_DIR.iterdir() if p.suffix.lower() in IMG_EXT)
if not new_paths:
    print(f"No photos in {NEW_DIR.resolve()}: add some photos of concrete structures and run this cell again.")
else:
    model = YOLO(str(FINAL_WEIGHTS))
    report, maps = [], {}
    for p, r in zip(new_paths, model.predict([str(p) for p in new_paths], imgsz=FINAL_IMGSZ, conf=CONF, verbose=False, device=DEVICE, stream=True)):
        cm = result_to_classmap(r, r.orig_shape)
        maps[p.name] = cm
        row = {"photo": p.name}
        for cid, c in enumerate(CLASSES, start=1):
            row[f"{c} regions"] = int((r.boxes.cls == cid - 1).sum()) if len(r.boxes) else 0
            row[f"{c} area %"] = round(100 * float((cm == cid).mean()), 2)
        found = [c for c in CLASSES if row[f"{c} regions"] > 0]
        row["damage found"] = ", ".join(found) or "none"
        row["severity"] = max([SEVERITY[c] for c in found], default=0)
        row["damaged area %"] = round(100 * float((cm > 0).mean()), 2)
        report.append(row)
    report = pd.DataFrame(report).sort_values(["severity", "damaged area %"], ascending=False).reset_index(drop=True)
    report.index = report.index + 1
    report.index.name = "priority"
    report.to_csv(FIG_DIR / "inspection_report.csv")
    display(report[["photo", "damage found", "severity", "damaged area %"]])

    top = report.head(8)
    cols = min(4, len(top)); rows_n = int(np.ceil(len(top) / cols))
    fig, axes = plt.subplots(rows_n, cols, figsize=(4.2 * cols, 4 * rows_n), squeeze=False)
    for a in axes.ravel():
        a.axis("off")
    for a, (prio, r) in zip(axes.ravel(), top.iterrows()):
        a.imshow(overlay(cv2.imread(str(NEW_DIR / r["photo"])), maps[r["photo"]]))
        a.set_title(f"#{prio}  {r['photo']}\n{r['damage found']}\n{r['damaged area %']:.1f}% of the photo", fontsize=9)
    fig.legend(handles=LEGEND, loc="upper center", ncol=5, bbox_to_anchor=(0.5, 1.0))
    fig.suptitle(f"Inspection list: new photos ranked by severity and damaged area (conf ≥ {CONF})", fontsize=13, y=1.04)
    fig.tight_layout()
    save_fig(fig, "10_application_inspection_list.png", "application.png")
    plt.show()

*Reading (fill in):* does the ranking make sense to you when you look at the photos? Which photo would you send an engineer to first, and does the model agree? Note any photo where the model clearly fails (other materials, painted surfaces, very dark photos).

## 8. Save, reload and use

The Ultralytics weights file contains the architecture **and** the class names, so `models/model.pt` is all a user needs. We also save the training log (for the training-curve figure) and a small `config.json` with the image size and the recommended threshold.

In [ ]:
shutil.copy(FINAL_WEIGHTS, MODEL_DIR / "model.pt")
shutil.copy(final["results"], MODEL_DIR / "training_log.csv")
config = {"classes": CLASSES, "imgsz": FINAL_IMGSZ, "recommended_conf": CONF, "base_model": "yolo26n-seg.pt",
          "train_minutes": round(float(final["minutes"]), 1) if final["minutes"] == final["minutes"] else None,
          "ultralytics": ultralytics.__version__, "torch": torch.__version__}
(MODEL_DIR / "config.json").write_text(json.dumps(config, indent=2))
print(json.dumps(config, indent=2))

In [ ]:
# As a user would: a new object, from the saved file, on one test photo
reloaded = YOLO(str(MODEL_DIR / "model.pt"))
print("Classes stored in the weights:", reloaded.names)
n_params = sum(p.numel() for p in reloaded.model.parameters())
print(f"Parameters: {n_params:,}")
example = YOLO_DIR / "images" / "test" / f"{show[0]}.jpg"
res = reloaded.predict(str(example), imgsz=FINAL_IMGSZ, conf=CONF, verbose=False)[0]
print(f"{example.name}: {len(res.boxes)} regions:",
      [f"{reloaded.names[int(c)]} {float(s):.2f}" for c, s in zip(res.boxes.cls, res.boxes.conf)])
plt.figure(figsize=(7, 5)); plt.imshow(res.plot()[..., ::-1]); plt.axis("off")
plt.title(f"Reloaded model on {example.name}"); plt.show()

### 8.1 Numbers for the README and the model card

The cell writes the result tables as Markdown into `figures/results_summary.md`, ready to paste into the README and the model card.

In [ ]:
def md_table(df, index=True):
    d = df.reset_index() if index else df
    fmt = lambda v: "–" if isinstance(v, float) and np.isnan(v) else (f"{v:.3f}" if isinstance(v, float) else str(v))
    lines = ["| " + " | ".join(map(str, d.columns)) + " |", "|" + "---|" * len(d.columns)]
    lines += ["| " + " | ".join(fmt(v) for v in row) + " |" for row in d.itertuples(index=False)]
    return "\n".join(lines)

text = (f"## Test set ({len(test_ids)} photos), recommended threshold {CONF}, image size {FINAL_IMGSZ}, {n_params:,} parameters\n\n"
        + md_table(overall.T.rename_axis("Model")) + "\n\n## Per class (test)\n\n" + md_table(per_class_test, index=False)
        + "\n\n## Experiment (validation)\n\n" + md_table(exp, index=False)
        + "\n\n## Data: images and instances per split\n\n" + md_table(summary.rename_axis("split")) + "\n")
(FIG_DIR / "results_summary.md").write_text(text)
print(text)

## 9. Conclusions

*Fill this in with your own reading, using the numbers printed above.* Guiding questions:

**Key findings** (3–6 bullet points, with numbers)
- How good is it overall? Mask mAP50 = … and mAP50-95 = … on the test set; the pretrained model scores 0 (no damage classes).
- Cracks: pixel IoU … for the model vs. … for the rule-based detector.
- Best and worst class: … (AP50 …) vs. … (AP50 …). Is the weakest one also the rarest?
- The experiment: image size 960 vs. 640 changed validation mask mAP50-95 by …, at … × the training time.
- At the recommended threshold (…), the mean recall is …: the model misses about … in … damage regions.

**Limitations and biases**
- **Data:** 2,119 photos collected from public datasets, mostly close-ups of damage; few photos of intact concrete, so false alarms on undamaged surfaces are probably underestimated. Rebar exposure and crushing have fewer than 100 instances each.
- **Split:** we grouped near-duplicate photos, but photos of the **same structure** taken from different viewpoints can still be in different splits, which may make the test score optimistic.
- **Labels:** regions, not true instances; a crack network is one polygon; exposure vs. corrosion is a judgement call.
- **Image conditions:** sensitive to scale (thin cracks), lighting, wet surfaces, paint, graffiti and other materials (masonry, steel).
- **Not for:** structural safety decisions or load-capacity assessment on its own, crack-width measurement, other materials, or replacing an inspection by an engineer. It is a **screening tool** that tells a human which photos to look at first.

## 10. Colab: download the results

On Colab, zip the figures and the model files, then download them together with this notebook (*File → Download → .ipynb*), and commit them to the repository. `models/model.pt` goes to Hugging Face only (it is in `.gitignore`).

In [ ]:
if IN_COLAB:
    out = Path("/content/day4_outputs.zip")
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
        for folder in [FIG_DIR, MODEL_DIR]:
            for p in folder.rglob("*"):
                if p.is_file():
                    z.write(p, p.resolve().relative_to(Path("..").resolve()))
    from google.colab import files
    files.download(str(out))